In [3]:
import pandas as pd
import numpy as np
import os

# Exploratory Analysis

In [ ]:
## Load the data with the target columns
# !cut -f 1,8,15,18,21,22,26,67,68,71 /home/camilababo/Documents/DNAquaIMG/BOLD_Public.29-Nov-2024/BOLD_Public.29-Nov-2024.tsv | gzip -c - > bold-selected.tsv.gz

In [2]:
# Load the data
path = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected.tsv"
bold = pd.read_csv(path, on_bad_lines="skip", sep="\t")

/tmp/ipykernel_25560/1221080447.py:3: DtypeWarning: Columns (1,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  bold = pd.read_csv(path, on_bad_lines="skip", sep="\t")


In [4]:
bold.head()

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code
0,AANIC003-10,BOLD:AAB9307,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P
1,AANIC002-10,BOLD:AAO2553,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCAGGTATAATTGGAACT...,658.0,COI-5P
2,AANIC058-10,BOLD:AAB9307,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATCTGAGCTGGTATAATTGGAACT...,658.0,COI-5P
3,AANIC061-10,BOLD:ACE8261,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P
4,AANIC062-10,BOLD:AAF6782,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P


In [4]:
bold.tail()

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code
17862992,ZYGMO209-10,BOLD:AAJ5548,Arthropoda,Zygaenidae,Adscita,Adscita obscura,NaN,AACACTTTACTTTATTTTTGGTATTTGATCAGGAATAGTTGGAACA...,658.0,COI-5P
17862993,ZYGMO290-10,BOLD:AAO1273,Arthropoda,Zygaenidae,Rhagades,Rhagades predotae,NaN,----------------------------------------------...,608.0,COI-5P
17862994,ZYGMO294-10,BOLD:AAO0371,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris minna,NaN,AACACTTTATTTTATTTTTGGAATTTGATCAGGAATAATTGGTACA...,658.0,COI-5P
17862995,ZYGMO295-10,BOLD:AAN9389,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris rjabovi,NaN,AACACTTTATTTTATCTTTGGAATCTGATCAGGAATAATTGGAACA...,658.0,COI-5P
17862996,ZYGMO296-10,BOLD:AAN9389,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris rjabovi,NaN,AACACTTTATTTTATCTTTGGAATCTGATCAGGAATAATTGGAACA...,658.0,COI-5P


In [17]:
# Check the total number of entries
bold.shape

(17862997, 10)

#### Check entries on 'Identification Method'

In [15]:
non_null_id = bold[bold['identification_method'] == 'Wikipidia']
non_null_id

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code
654690,AZSDS020-18,NaN,Arthropoda,Cerambycidae,NaN,NaN,Wikipidia,NaN,NaN,NaN


In [18]:
# Check the number of entries with COI-5P
count = bold['marker_code'].str.contains('COI-5P', na=False).sum()
int(count)

14790363

In [19]:
# Check the average length of entrie's nucleotide sequence
average_length = bold['nuc'].astype(str).apply(len).mean()
int(average_length)

610

In [13]:
# Check values with unique entries for 'identification method'
unique_idm = bold['identification_method'].unique()
np.savetxt("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/idm.txt",
           unique_idm, fmt='%s')

In [ ]:
# Check values with unique entries for 'marker code'
unique_mc = bold['marker_code'].unique()
np.savetxt("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/mc.txt",
           unique_mc, fmt='%s')

## Pre-processing

#### Filter out entries whose 'marker_code' is not 'COI-5P'

In [3]:
bold = bold[bold['marker_code'] == 'COI-5P']

#### Remove all entries whose sequence is inferior to 500 nucleotides

In [4]:
bold = bold[bold['nuc'].notna() & (bold['nuc_basecount'] > 500.0)]

#### Create column indicating highest taxon level available

In [5]:
bold['highest_tax_level'] = bold[['species', 'genus', 'family', 'phylum']].bfill(axis=1).iloc[:, 0]

#### Filter out entries whose presence of degenerate nucleotides is superior to 1% of the total sequence length

In [6]:
bold['N_count'] = bold['nuc'].str.count('N')

In [7]:
bold['prct_degenerate'] = (bold['N_count'] / bold['nuc_basecount']) * 100

In [8]:
bold = bold[bold['prct_degenerate'] <= 1]

#### Save the data

In [9]:
bold.shape

(14109605, 13)

In [10]:
bold.to_csv("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected-init-preprocess.tsv", sep="\t", index=False)

#### Taxonomic Harmonization

In [11]:
# Select highest taxonomic level column to run Tax. Harmonizer
bold_highest_tax = bold[['highest_tax_level']]
bold_highest_tax.rename(columns={'highest_tax_level': 'taxa'}, inplace=True)
bold_highest_tax.to_csv("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected-highest_tax.tsv", sep="\t", index=False)

In [18]:
bold_highest_tax.to_csv("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected-highest_tax.tsv", sep="\t", index=False)

In [13]:
# Transform the column into a list
list = bold_highest_tax['highest_tax_level'].tolist()
# CHeck the number of unique values
len(set(list))

379109

In [4]:
# Load the harmonized data
path = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected-highest_tax_harmonized.tsv"
bold_init_filt = pd.read_csv(path, on_bad_lines="skip", sep="\t")

In [27]:
bold_init_filt

,taxa,scientificName,rank,kingdom,phylum,class,order,family,genus
0,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia
1,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia
2,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia
3,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia
4,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia
...,...,...,...,...,...,...,...,...,...
14109600,Adscita obscura,"Adscita obscura (Zeller, 1847)",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Zygaenidae,Adscita
14109601,Rhagades predotae,"Rhagades predotae (Naufock, 1930)",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Zygaenidae,Rhagades
14109602,Zygaenoprocris minna,Zygaenoprocris minna,SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Zygaenidae,Zygaenoprocris
14109603,Zygaenoprocris rjabovi,Zygaenoprocris rjabovi,SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Zygaenidae,Zygaenoprocris


In [28]:
no_hit = bold_init_filt[bold_init_filt['scientificName'] == '-']

In [41]:
no_hit_taxa = no_hit['taxa'].to_list()
len(no_hit_taxa)

325734

In [46]:
(325734 / 14109605) * 100

2.308597582993996

In [43]:
no_hit_taxa_unique = no_hit['taxa'].unique()
len(no_hit_taxa_unique)

19458

In [45]:
with open("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/no_gbif_hit_taxa.txt", "w") as file:
    for entry in no_hit_taxa_unique:
        file.write(entry + "\n")

##### Merge pre-processed BOLD with Harmonized entries

In [1]:
# Merge bold and bold_init_filt
chunk_size = 10000
chunks = []

for start in range(0, len(bold), chunk_size):
    end = start + chunk_size
    chunk = bold.iloc[start:end]
    merged_chunk = pd.merge(chunk, bold_init_filt, left_on='highest_tax_level', right_on='taxa', how='left')
    chunks.append(merged_chunk)

bold_harmonized = pd.concat(chunks, ignore_index=True)


NameError: name 'bold' is not defined

In [78]:
bold_harmonized

NameError: name 'bold_harmonized' is not defined

##### Filter BOLD data package by OTL's Phylums

In [59]:
def get_unique_phylum_values(folder_path):
    """
    Recursively reads all .csv files in a folder (including subfolders),
    extracts unique values from the 'phylum' column, and returns a set of unique values.

    Parameters:
        folder_path (str): Path to the folder containing .csv files.

    Returns:
        set: A set of unique values from the 'phylum' column across all files.
    """
    unique_phyla = set()

    for root, _, files in os.walk(folder_path):
        for file_name in files:
            if file_name.endswith(".tsv"):
                file_path = os.path.join(root, file_name)

                try:
                    df = pd.read_csv(file_path, sep='\t')

                    if 'phylum' in df.columns:
                        # Add unique values from the 'phylum' column to the set
                        unique_phyla.update(df['phylum'].dropna().unique())
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")

    return unique_phyla

In [62]:
folder_path = "/home/camilababo/Documents/DNAquaIMG/countries-otls/harmonized"
unique_phyla = get_unique_phylum_values(folder_path)
unique_phyla.remove('-')

In [63]:
unique_phyla

{'Annelida',
 'Arthropoda',
 'Bryozoa',
 'Charophyta',
 'Chlorophyta',
 'Chordata',
 'Cnidaria',
 'Cyanobacteria',
 'Entoprocta',
 'Mollusca',
 'Nematoda',
 'Nematomorpha',
 'Nemertea',
 'Ochrophyta',
 'Platyhelminthes',
 'Porifera',
 'Rhodophyta',
 'Tardigrada',
 'Tracheophyta'}

In [17]:
# Check values with unique entries for 'identification method'
unique_idm = bold['identification_method'].unique()
np.savetxt("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/idm.txt",
           unique_idm, fmt='%s')